<a href="https://colab.research.google.com/github/Rafak22/python/blob/main/Another_copy_of_SQL_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install langchain  langgraph  langchain-community langchain_openrouter
%pip install -U "langchain[huggingface]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.4/396.4 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google

In [16]:
import os
try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("API_Key")
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")
except ImportError:
    # Load `.env` file (locally)
    from dotenv import load_dotenv
    load_dotenv(override=True)

Phi Model below will struggle to call the tools needed to generate the SQL queries necessary for the task

In [17]:
import os
from langchain.chat_models import init_chat_model

# model = init_chat_model(
#     "Qwen2.5-Coder-7B-Instruct",
#     model_provider="huggingface",
#     temperature=0.1,
#     max_tokens=4096,
# )

# model = init_chat_model(
#     "microsoft/Phi-3-mini-4k-instruct",
#     model_provider="huggingface",
#     temperature=0.1,
#     max_tokens=4096,
# )

Try this model for proper Tool calling

In [18]:
import os
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "auto",
    model_provider="openrouter",
)

In [19]:
import requests, pathlib

url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
local_path = pathlib.Path("Chinook.db")

if local_path.exists():
    print(f"{local_path} already exists, skipping download.")
else:
    response = requests.get(url)
    if response.status_code == 200:
        local_path.write_bytes(response.content)
        print(f"File downloaded and saved as {local_path}")
    else:
        print(f"Failed to download the file. Status code: {response.status_code}")

File downloaded and saved as Chinook.db


In [20]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

print(f"Dialect: {db.dialect}")
print(f"Available tables: {db.get_usable_table_names()}")
print(f'Sample output: {db.run("SELECT * FROM Artist LIMIT 5;")}')

Dialect: sqlite
Available tables: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']
Sample output: [(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]


In [21]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=model)

tools = toolkit.get_tools()

for tool in tools:
    print(f"{tool.name}: {tool.description}\n")

sql_db_query: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables: Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



In [22]:
system_prompt = """
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

If the user question requires joining tables and/or using lookup tables. Make sure
you check the entity names the user provided to match them with the lookup table
if needed. If they don't match, try to find the nearest match and use it instead.

When filtering by a string or name (like Genre.Name, Track.Name, or Artist.Name),
DO NOT use exact matching (=). Instead, always use the LIKE operator with wildcards to do a fuzzy match.
For example, instead of WHERE G.Name = 'Sci Fi', use WHERE G.Name LIKE '%Sci Fi%'.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables. Please run the query
and get the results to the user. Do not create the queries without running them as
the user is only interested in the answer to their question
""".format(
    dialect=db.dialect,
    top_k=5,
)

In [23]:
from langchain.agents import create_agent


agent = create_agent(
    model,
    tools,
    system_prompt=system_prompt,
)

In [25]:
import uuid

# question = "Which genre on average has the longest tracks?"
# question = "how many tracks do we have in the Sci Fi & Fantasy genre"
# question = "are there any genres that might be duplicated in the genre table? even if the name is not exactly the same?"
# This will yield to 0 results despite having tracks on the Sci Fi & Fantasy genre
question = "Group albums by the first letter of their title and count them."

config = {"configurable": {"thread_id": f"{uuid.uuid4()}"}}

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Group albums by the first letter of their title and count them.
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (tool_sql_db_list_tables_g2XPhLkSB4j0zQMfXsYj)
 Call ID: tool_sql_db_list_tables_g2XPhLkSB4j0zQMfXsYj
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================

The user wants to group albums by the first letter of their title and count them. I need to query the 'Album' table. First, I will check the schema of the 'Album' table to understand its columns.
Tool Calls:
  sql_db_schema (tool_sql_db_schema_fdNpr67Liertiu1AvkU9)
 Call ID: tool_sql_db_schema_fdNpr67Liertiu1AvkU9
  Args:
    table_n

In [ ]:
# from langchain.agents import create_agent
# from langchain.agents.middleware import HumanInTheLoopMiddleware
# from langgraph.checkpoint.memory import InMemorySaver


# agent = create_agent(
#     model,
#     tools,
#     system_prompt=system_prompt,
#     middleware=[
#         HumanInTheLoopMiddleware(
#             interrupt_on={"sql_db_query": True},
#             description_prefix="Tool execution pending approval",
#         ),
#     ],
#     checkpointer=InMemorySaver(),
# )

In [ ]:
# question = "Which genre on average has the longest tracks?"
# config = {"configurable": {"thread_id": "1"}}

# for step in agent.stream(
#     {"messages": [{"role": "user", "content": question}]},
#     config,
#     stream_mode="values",
# ):
#     if "__interrupt__" in step:
#         print("INTERRUPTED:")
#         interrupt = step["__interrupt__"][0]
#         for request in interrupt.value["action_requests"]:
#             print(request["description"])
#     elif "messages" in step:
#         step["messages"][-1].pretty_print()
#     else:
#         pass

In [ ]:
# # Example of invoking the agent in one step
# question_invoke = "how many tracks are there in the sci-fi genre"
# config_invoke = {"configurable": {"thread_id": "2"}} # Use a different thread_id for a new conversation

# result = agent.invoke(
#     {"messages": [{"role": "user", "content": question_invoke}]},
#     config_invoke
# )

# # The result will contain a list of messages, the final answer is usually the last one.
# print("Invoked Agent Result:")
# result["messages"][-1].pretty_print()

In [ ]:
# from langgraph.types import Command

# for step in agent.stream(
#     Command(resume={"decisions": [{"type": "approve"}]}),
#     config,
#     stream_mode="values",
# ):
#     if "messages" in step:
#         step["messages"][-1].pretty_print()
#     if "__interrupt__" in step:
#         print("INTERRUPTED:")
#         interrupt = step["__interrupt__"][0]
#         for request in interrupt.value["action_requests"]:
#             print(request["description"])
#     else:
#         pass